# Preparation des donnees - pipeline complet (etapes 1 a 6)

Notebook unique, commente et reproductible, qui enchaine dans l'ordre les 6 etapes de preparation du dataset :
chargement/nettoyage initial, traduction camerounaise, enrichissement camerounais, nettoyage avance,
traitement des valeurs manquantes/aberrantes, puis encodage/normalisation.

**Entree** : `data/raw/Loan_Default.csv` (dataset Kaggle "Loan Default Dataset", yasserh, ~148 670 lignes)

**Sortie finale** : `data/processed/Loan_Default_Cameroun_Encode.csv` (19 features ML, pret pour la modelisation en Semaine 3)

A executer de haut en bas, en un seul passage, dans un kernel fraichement demarre : c'est ce qui prouve la
reproductibilite du pipeline (livrable Semaine 2 / vendredi 7 aout, cf. `docs/Analyse_et_Plan_Projet.docx` section 4).
Chaque etape ci-dessous correspond a un script individuel du dossier `scripts/` (01 a 06), garde separement
pour un usage/debug unitaire ; ce notebook est la version consolidee servant de preuve de bout-en-bout.

A executer en local (VS Code / Jupyter) ou en copiant les cellules dans Google Colab.

---
## Etape 1 - Chargement et nettoyage initial
*(source : `scripts/01_chargement_nettoyage.py`)*

## Semaine 1 - Etape 1 : Chargement et nettoyage initial
Dataset attendu : Kaggle "Loan Default Dataset" (yasserh), ~148 670 lignes.
A executer en local (VS Code / Jupyter) ou en copiant les cellules dans Google Colab.

In [1]:
import os
import sys

import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Chaque membre du groupe doit avoir son propre token Kaggle (gratuit) :
    # https://www.kaggle.com/settings -> API -> "Create New Token" (telecharge kaggle.json)
    BASE_DIR = "/content/ProjetScoringCredit"
    os.makedirs(f"{BASE_DIR}/data/raw", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/data/processed", exist_ok=True)
    if not os.path.exists(f"{BASE_DIR}/data/raw/Loan_Default.csv"):
        from google.colab import files
        print("Uploadez votre kaggle.json (Kaggle > Settings > API > Create New Token) :")
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.system("cp kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json")
        os.system("pip install -q kaggle")
        os.system(
            f"kaggle datasets download -d yasserh/loan-default-dataset "
            f"-p {BASE_DIR}/data/raw --unzip"
        )
else:
    BASE_DIR = ".."

RAW_PATH = f"{BASE_DIR}/data/raw/Loan_Default.csv"

df = pd.read_csv(RAW_PATH)
print("Dimensions :", df.shape)
print("\nColonnes trouvees :")
print(list(df.columns))

Dimensions : (148670, 34)

Colonnes trouvees :
['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type', 'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type', 'Status', 'dtir1']


### Verification des colonnes attendues
Si le fichier telecharge correspond au dataset "Loan Default Dataset" de yasserh,
on doit retrouver approximativement ces colonnes. On compare pour detecter un ecart
(nom different, colonnes manquantes/en plus) avant d'aller plus loin.

In [2]:
colonnes_attendues = [
    "ID", "year", "loan_limit", "Gender", "approv_in_adv", "loan_type",
    "loan_purpose", "Credit_Worthiness", "open_credit", "business_or_commercial",
    "loan_amount", "rate_of_interest", "Interest_rate_spread", "Upfront_charges",
    "term", "Neg_ammortization", "interest_only", "lump_sum_payment",
    "property_value", "construction_type", "occupancy_type", "Secured_by",
    "total_units", "income", "credit_type", "Credit_Score",
    "co-applicant_credit_type", "age", "submission_of_application", "LTV",
    "Region", "Security_Type", "Status", "dtir1",
]

manquantes = [c for c in colonnes_attendues if c not in df.columns]
en_plus = [c for c in df.columns if c not in colonnes_attendues]
print("Colonnes attendues manquantes :", manquantes)
print("Colonnes presentes non attendues :", en_plus)

Colonnes attendues manquantes : []
Colonnes presentes non attendues : []


### Types et valeurs manquantes

In [3]:
print(df.dtypes)
print("\nValeurs manquantes par colonne (top 15) :")
print(df.isna().sum().sort_values(ascending=False).head(15))

ID                             int64
year                           int64
loan_limit                       str
Gender                           str
approv_in_adv                    str
loan_type                        str
loan_purpose                     str
Credit_Worthiness                str
open_credit                      str
business_or_commercial           str
loan_amount                    int64
rate_of_interest             float64
Interest_rate_spread         float64
Upfront_charges              float64
term                         float64
Neg_ammortization                str
interest_only                    str
lump_sum_payment                 str
property_value               float64
construction_type                str
occupancy_type                   str
Secured_by                       str
total_units                      str
income                       float64
credit_type                      str
Credit_Score                   int64
co-applicant_credit_type         str
a

### Valeurs manquantes deguisees
1260 lignes ont income == 0 (jamais negatif). Un revenu mensuel exactement nul
pour un demandeur de credit n'est pas plausible : c'est tres probablement un
encodage de valeur manquante par 0 plutot qu'un NaN explicite.
On le convertit en NaN pour que l'imputation de la Semaine 2 (mediane) le
traite comme les 9150 valeurs deja manquantes, plutot que de laisser un "0
FCFA de revenu mensuel" une fois la mise a l'echelle camerounaise appliquee
(risque de division par zero dans les ratios derives, ex. ratio_endettement).

In [4]:
n_income_zero = (df["income"] == 0).sum()
print(f"Lignes avec income == 0 (traitees comme manquantes) : {n_income_zero}")
df.loc[df["income"] == 0, "income"] = pd.NA

Lignes avec income == 0 (traitees comme manquantes) : 1260


### Doublons

In [5]:
n_doublons = df.duplicated().sum()
print(f"Doublons exacts detectes : {n_doublons}")
df = df.drop_duplicates()

if "ID" in df.columns:
    n_doublons_id = df.duplicated(subset="ID").sum()
    print(f"Doublons sur la colonne ID : {n_doublons_id}")
    df = df.drop_duplicates(subset="ID")

print("Dimensions apres suppression des doublons :", df.shape)

Doublons exacts detectes : 0


Doublons sur la colonne ID : 0
Dimensions apres suppression des doublons : (148670, 34)


### Sauvegarde intermediaire (avant traduction camerounaise)

In [6]:
OUTPUT_PATH = f"{BASE_DIR}/data/processed/loan_default_clean.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Fichier nettoye sauvegarde : {OUTPUT_PATH}")

Fichier nettoye sauvegarde : ../data/processed/loan_default_clean.csv


---
## Etape 2 - Traduction camerounaise du dataset
*(source : `scripts/02_traduction_camerounaise.py`)*

## Semaine 1 - Etape 2 : Traduction camerounaise du dataset
Entree  : ../data/processed/loan_default_clean.csv (sortie du script 01)
Sortie  : ../data/processed/Loan_Default_Cameroun.csv  (livrable Semaine 1 d'Aristide)

Objectif : renommer les colonnes en francais et adapter les valeurs a un contexte
camerounais (montants en FCFA, regions du Cameroun, libelles en francais).
Les variables METIER specifiques (tontine, mobile money, secteur informel, etc.)
seront ajoutees par Marie-Therese en Semaine 2 : ce script ne fait QUE la traduction
du dataset original, pas l'enrichissement.

In [7]:
import sys

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/ProjetScoringCredit" if IN_COLAB else ".."

INPUT_PATH = f"{BASE_DIR}/data/processed/loan_default_clean.csv"
OUTPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun.csv"

# IMPORTANT : ce n'est PAS un taux de change USD/FCFA. Ce dataset represente des prets
# immobiliers americains ; ses revenus/montants sont structurellement bien superieurs
# a un contexte camerounais. Un taux de change brut (~600) donnerait un revenu mensuel
# median de ~3,45M FCFA, alors que le salaire moyen FORMEL au Cameroun est ~120 000
# FCFA/mois et le SMIG ~60 000 FCFA/mois (sources : africarrieres.com, infospratiques.cm).
#
# FACTEUR_ECHELLE est donc calibre pour que le revenu mensuel median obtenu (5760 x
# FACTEUR_ECHELLE) se rapproche du salaire moyen formel camerounais, plutot que de
# refleter un taux de change reel. Applique de facon UNIFORME a toutes les colonnes
# monetaires, il preserve les ratios internes du dataset (LTV, ratio_endettement,
# pret/revenu), qui restent la variable predictive interessante pour le modele.
# -> A documenter explicitement comme hypothese methodologique dans le rapport.
FACTEUR_ECHELLE = 20

df = pd.read_csv(INPUT_PATH)

### 1. Renommage des colonnes (anglais -> francais)

In [8]:
RENOMMAGE_COLONNES = {
    "ID": "id_client",
    "year": "annee",
    "loan_limit": "plafond_pret",
    "Gender": "genre",
    "approv_in_adv": "approbation_anticipee",
    "loan_type": "type_pret",
    "loan_purpose": "objet_pret",
    "Credit_Worthiness": "solvabilite",
    "open_credit": "credit_ouvert",
    "business_or_commercial": "usage_professionnel",
    "loan_amount": "montant_pret_fcfa",
    "rate_of_interest": "taux_interet",
    "Interest_rate_spread": "ecart_taux_interet",
    "Upfront_charges": "frais_initiaux_fcfa",
    "term": "duree_mois",
    "Neg_ammortization": "amortissement_negatif",
    "interest_only": "interet_seul",
    "lump_sum_payment": "paiement_forfaitaire",
    "property_value": "valeur_bien_fcfa",
    "construction_type": "type_construction",
    "occupancy_type": "type_occupation",
    "Secured_by": "garanti_par",
    "total_units": "nombre_unites",
    "income": "revenu_mensuel_fcfa",
    "credit_type": "type_credit_bureau",
    "Credit_Score": "score_credit_bureau",
    "co-applicant_credit_type": "type_credit_coemprunteur",
    "age": "tranche_age",
    "submission_of_application": "mode_soumission",
    "LTV": "ratio_pret_valeur",
    "Region": "region_cameroun",
    "Security_Type": "type_garantie",
    "Status": "statut_remboursement",
    "dtir1": "ratio_endettement",
}

df = df.rename(columns=RENOMMAGE_COLONNES)

colonnes_traduites = [v for k, v in RENOMMAGE_COLONNES.items() if k in df.columns or v in df.columns]
print(f"{len(colonnes_traduites)} colonnes renommees en francais.")

34 colonnes renommees en francais.


### 2. Mise a l'echelle des montants (contexte camerounais)
Facteur unique applique uniformement a toutes les colonnes monetaires pour
preserver leurs ratios internes (LTV, ratio_endettement, pret/revenu).

In [9]:
for col in ["montant_pret_fcfa", "frais_initiaux_fcfa", "valeur_bien_fcfa", "revenu_mensuel_fcfa"]:
    if col in df.columns:
        df[col] = (df[col] * FACTEUR_ECHELLE).round(0)

### 3. Traduction des valeurs categorielles

In [10]:
MAPPINGS_VALEURS = {
    "plafond_pret": {"cf": "Conforme", "ncf": "Non_conforme"},
    "genre": {
        "Male": "Homme",
        "Female": "Femme",
        "Joint": "Conjoint",
        "Sex Not Available": "Non_specifie",
    },
    "approbation_anticipee": {"pre": "Oui", "nopre": "Non"},
    "type_pret": {"type1": "Type_1", "type2": "Type_2", "type3": "Type_3"},
    # p3 recadre volontairement "home improvement" (source immobiliere US) en
    # categorie generale d'investissement professionnel/agricole : l'application
    # cible le credit en general (besoins varies), pas le credit immobilier.
    "objet_pret": {"p1": "Achat", "p2": "Refinancement", "p3": "Investissement_activite", "p4": "Autre"},
    "solvabilite": {"l1": "Standard", "l2": "Sous_standard"},
    "credit_ouvert": {"nopc": "Non", "opc": "Oui"},
    "usage_professionnel": {"nob/c": "Non", "b/c": "Oui"},
    "amortissement_negatif": {"neg_amm": "Oui", "not_neg": "Non"},
    "interet_seul": {"int_only": "Oui", "not_int": "Non"},
    "paiement_forfaitaire": {"lpsm": "Oui", "not_lpsm": "Non"},
    "type_occupation": {"pr": "Residence_principale", "sr": "Residence_secondaire", "ir": "Investissement"},
    "garanti_par": {"home": "Logement", "land": "Terrain"},
    "mode_soumission": {"to_inst": "En_agence", "not_inst": "En_ligne"},
    # NB: le dataset source contient la faute de frappe "Indriect" (pas "Indirect")
    "type_garantie": {"direct": "Directe", "Indriect": "Indirecte"},
    "statut_remboursement": {0: "Rembourse", 1: "Defaut"},
    # Regions US du dataset source -> regions du Cameroun (mapping fixe 1:1 pour la trace)
    "region_cameroun": {
        "south": "Littoral",
        "North": "Centre",
        "central": "Ouest",
        "North-East": "Nord",
    },
}

for col, mapping in MAPPINGS_VALEURS.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)

### 4. Verification finale

In [11]:
print("Dimensions finales :", df.shape)
print("\nApercu :")
print(df.head(3).T)

nb_colonnes_traduites = sum(1 for c in RENOMMAGE_COLONNES.values() if c in df.columns)
print(f"\nColonnes traduites presentes dans le fichier final : {nb_colonnes_traduites} (objectif >= 15)")

Dimensions finales : (148670, 34)

Apercu :
                                             0                     1  \
id_client                                24890                 24891   
annee                                     2019                  2019   
plafond_pret                          Conforme              Conforme   
genre                             Non_specifie                 Homme   
approbation_anticipee                      Non                   Non   
type_pret                               Type_1                Type_2   
objet_pret                               Achat                 Achat   
solvabilite                           Standard              Standard   
credit_ouvert                              Non                   Non   
usage_professionnel                        Non                   Oui   
montant_pret_fcfa                      2330000               4130000   
taux_interet                               NaN                   NaN   
ecart_taux_interet  

### 5. Sauvegarde du livrable Semaine 1

In [12]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Livrable sauvegarde : {OUTPUT_PATH}")

Livrable sauvegarde : ../data/processed/Loan_Default_Cameroun.csv


---
## Etape 3 - Enrichissement camerounais (variables contextuelles)
*(source : `scripts/03_enrichissement_camerounais.py`)*

## Semaine 1 - Etape 3 : Assainissement et enrichissement camerounais
Entree  : ../data/processed/Loan_Default_Cameroun.csv (sortie du script 02)
          ../data/processed/secteur_activite_niveau_education.csv (variables
          de Marie-Therese conservees telles quelles, voir section 2)
Sortie  : ../data/processed/Loan_Default_Cameroun_Enrichi.csv (livrable Semaine 1)

Objectif : appliquer les decisions actees le 2026-07-29 (voir
docs/Analyse_et_Plan_Projet.docx, section 3) sur les 8 variables camerounaises
ajoutees par Marie-Therese en Semaine 2 initiale, et sur le perimetre metier
(credit en general, pas credit immobilier) :
- supprimer les colonnes de fuite/redondance (statut_remboursement_label,
  zone_geographique) ;
- regenerer 3 variables bruitees en les conditionnant sur des colonnes reelles
  deja correlees a la cible (secteur_activite, region_cameroun), au lieu d'un
  tirage uniforme sans rapport avec le risque de defaut ;
- retirer les colonnes specifiques au credit immobilier (fuite + hors perimetre
  produit) ;
- ne PAS recreer categorie_risque ici : c'est une sortie calculee par l'app a
  partir du score du modele, jamais une variable d'entree.

In [13]:
import sys

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/ProjetScoringCredit" if IN_COLAB else ".."

INPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun.csv"
LOOKUP_PATH = f"{BASE_DIR}/data/processed/secteur_activite_niveau_education.csv"
OUTPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Enrichi.csv"

GRAINE_ALEATOIRE = 42
rng = np.random.default_rng(GRAINE_ALEATOIRE)

df = pd.read_csv(INPUT_PATH)
print("Dimensions en entree :", df.shape)

Dimensions en entree : (148670, 34)


### 1. Suppression des colonnes specifiques au credit immobilier
`valeur_bien_fcfa` est manquant pour 10.2% des lignes, et ces lignes ont un
taux de defaut de 100.0% (contre 16.1% quand la valeur est presente) : un
artefact de collecte du dataset source (la valorisation du bien s'arrete
probablement d'etre enregistree une fois le pret en defaut/saisie), pas un
vrai signal de risque transferable a un nouveau demandeur camerounais.
Ces colonnes sont de toute facon specifiques a un credit immobilier, hors du
perimetre produit (credit en general) : leur suppression resout a la fois la
fuite et le desalignement de perimetre. Elles restent disponibles dans
Loan_Default_Cameroun.csv pour un futur module optionnel "credit immobilier"
hors perimetre V1.

In [14]:
COLONNES_IMMOBILIER = [
    "valeur_bien_fcfa",
    "ratio_pret_valeur",
    "type_construction",
    "type_occupation",
    "nombre_unites",
    "garanti_par",
    "type_garantie",
]

colonnes_a_retirer = [c for c in COLONNES_IMMOBILIER if c in df.columns]
df = df.drop(columns=colonnes_a_retirer)
print(f"{len(colonnes_a_retirer)} colonnes immobilieres retirees : {colonnes_a_retirer}")

7 colonnes immobilieres retirees : ['valeur_bien_fcfa', 'ratio_pret_valeur', 'type_construction', 'type_occupation', 'nombre_unites', 'garanti_par', 'type_garantie']


### 2. Ajout des variables camerounaises conservees telles quelles
`secteur_activite` et `niveau_education` (variables de Marie-Therese) ne sont
pas regenerees : l'audit du 2026-07-29 a confirme que `secteur_activite`
porte un vrai gradient de risque (23.5-24.2% pour les 3 secteurs dominants vs
34-37% pour Commerce/Negoce, Profession liberale, Artisanat), et
`niveau_education` est conservee pour la description du profil dans le
formulaire (elle n'entre pas dans les variables du modele ML, decide en
feature selection en Semaine 2, pas dans ce script).

In [15]:
lookup = pd.read_csv(LOOKUP_PATH)
n_avant = len(df)
df = df.merge(lookup, on="id_client", how="left")
assert len(df) == n_avant, "Le merge a modifie le nombre de lignes"
assert df["secteur_activite"].isna().sum() == 0, "secteur_activite manquant apres merge"
print("secteur_activite / niveau_education ajoutees. Repartition secteur_activite :")
print(df["secteur_activite"].value_counts())

secteur_activite / niveau_education ajoutees. Repartition secteur_activite :
secteur_activite
SalariÃ© formel         46773
Agriculture             45718
Petit commerce          45367
Commerce/NÃ©goce         5877
Profession libÃ©rale     3842
Artisanat                1093
Name: count, dtype: int64


### 3. Regeneration des 3 variables bruitees
L'audit du 2026-07-29 a montre que ces 3 variables, telles que generees
initialement, etaient quasi non correlees a la cible (bruit ~aleatoire),
malgre une documentation annoncant des "distributions realistes". On les
regenere ici conditionnees sur des colonnes reelles deja correlees au risque
(`secteur_activite`, `region_cameroun`), sans jamais lire la cible.

Limite assumee (verifiee le 2026-07-30, decision d'Aristide) : le
regroupement de `membre_tontine` (secteur informel/traditionnel : Agriculture,
Petit commerce, Artisanat vs secteur formel/liberal : Salarie formel,
Commerce/Negoce, Profession liberale) reflete une segmentation socio-
economique reelle, mais melange des secteurs a risque de defaut oppose
(Artisanat est le secteur le PLUS risque - 37.3% - tout en etant classe dans
le groupe "forte tontine" avec Agriculture/Petit commerce, qui sont les
secteurs les MOINS risques). Consequence verifiee : le taux de defaut par
`membre_tontine` reste quasi plat (24.8% Non vs 24.5% Oui), tout comme pour
`activite_saisonniere` et `utilisation_mobile_money`. Choix delibere : ne pas
reforcer artificiellement une correlation avec la cible pour ces 3 variables
- elles restent des signaux socio-economiques plausibles pour le profil
camerounais, meme faiblement predictifs du defaut. A documenter comme limite
honnete dans le rapport plutot que de forcer un signal.

In [16]:
PROBA_TONTINE = {
    "Agriculture": 0.55,
    "Petit commerce": 0.55,
    "Artisanat": 0.55,
    "Salarié formel": 0.25,
    "Commerce/Négoce": 0.25,
    "Profession libérale": 0.25,
}
proba_tontine = df["secteur_activite"].map(PROBA_TONTINE).fillna(0.40)
df["membre_tontine"] = np.where(rng.random(len(df)) < proba_tontine, "Oui", "Non")

PROBA_SAISONNIERE = {
    "Agriculture": 0.85,
    "Petit commerce": 0.35,
    "Commerce/Négoce": 0.35,
}
proba_saisonniere = df["secteur_activite"].map(PROBA_SAISONNIERE).fillna(0.10)
df["activite_saisonniere"] = np.where(rng.random(len(df)) < proba_saisonniere, "Oui", "Non")

# Mobile money : penetration plus forte en zone urbaine (Littoral/Centre) et
# chez les demandeurs plus instruits. Le mapping region -> urbanite est le
# meme que l'ancienne colonne zone_geographique (deterministe, donc pas besoin
# de la stocker comme colonne a part - voir section 4).
BASE_REGION_MOBILE_MONEY = {
    "Littoral": 0.75,
    "Centre": 0.75,
    "Ouest": 0.55,
    "Nord": 0.30,
}
AJUSTEMENT_EDUCATION_MOBILE_MONEY = {
    "Sans diplôme": -0.15,
    "Primaire": -0.05,
    "Secondaire": 0.05,
    "Supérieur": 0.15,
}
base_region = df["region_cameroun"].map(BASE_REGION_MOBILE_MONEY).fillna(0.50)
ajustement_education = df["niveau_education"].map(AJUSTEMENT_EDUCATION_MOBILE_MONEY).fillna(0.0)
proba_mobile_money = (base_region + ajustement_education).clip(0.05, 0.97)
df["utilisation_mobile_money"] = np.where(rng.random(len(df)) < proba_mobile_money, "Oui", "Non")

print("Variables regenerees : membre_tontine, activite_saisonniere, utilisation_mobile_money")

Variables regenerees : membre_tontine, activite_saisonniere, utilisation_mobile_money


### 4. Colonnes volontairement absentes de ce livrable
- `statut_remboursement_label` : doublon exact de la cible `statut_remboursement`
  (fuite pure). `statut_remboursement` est deja lisible en francais
  ("Rembourse"/"Defaut"), donc aucune reconstruction n'est necessaire pour
  l'affichage dans l'app.
- `zone_geographique` : redondante a 100% avec `region_cameroun` (mapping
  deterministe 1:1, aucune variance intra-region).
- `categorie_risque` : ce n'est pas une variable d'entree. Elle sera calculee
  par l'app a partir du score predit par le modele (>=70 Faible, 55-69
  Modere, 40-54 Eleve, <40 Tres haut risque), en Semaine 4.

### 5. Verification finale

In [17]:
print("Dimensions finales :", df.shape)

print("\nTaux de defaut par secteur_activite (doit conserver le gradient) :")
print(df.groupby("secteur_activite")["statut_remboursement"].apply(lambda s: (s == "Defaut").mean()).sort_values())

print("\nTaux de defaut par membre_tontine (attendu quasi plat, limite assumee - voir section 3) :")
print(df.groupby("membre_tontine")["statut_remboursement"].apply(lambda s: (s == "Defaut").mean()))

for col in COLONNES_IMMOBILIER:
    assert col not in df.columns, f"{col} n'aurait pas du etre presente"
for col in ["statut_remboursement_label", "zone_geographique", "categorie_risque"]:
    assert col not in df.columns, f"{col} n'aurait pas du etre presente"

print("\nValeurs manquantes restantes (top 10) :")
print(df.isna().sum().sort_values(ascending=False).head(10))

Dimensions finales : (148670, 32)

Taux de defaut par secteur_activite (doit conserver le gradient) :
secteur_activite
Petit commerce          0.235193
Agriculture             0.238527
SalariÃ© formel         0.241742
Commerce/NÃ©goce        0.340140
Profession libÃ©rale    0.351379
Artisanat               0.373285
Name: statut_remboursement, dtype: float64

Taux de defaut par membre_tontine (attendu quasi plat, limite assumee - voir section 3) :


membre_tontine
Non    0.248025
Oui    0.244817
Name: statut_remboursement, dtype: float64

Valeurs manquantes restantes (top 10) :
frais_initiaux_fcfa      39642
ecart_taux_interet       36639
taux_interet             36439
ratio_endettement        24121
revenu_mensuel_fcfa      10410
plafond_pret              3344
approbation_anticipee      908
tranche_age                200
mode_soumission            200
objet_pret                 134
dtype: int64


### 6. Sauvegarde du livrable Semaine 1

In [18]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Livrable sauvegarde : {OUTPUT_PATH}")

Livrable sauvegarde : ../data/processed/Loan_Default_Cameroun_Enrichi.csv


---
## Etape 4 - Nettoyage avance (retrait des colonnes exclues)
*(source : `scripts/04_nettoyage_avance.py`)*

## Semaine 1 (ajustement) - Etape 4 : nettoyage avance avant la Semaine 2
Entree  : ../data/processed/Loan_Default_Cameroun_Enrichi.csv (sortie du script 03)
Sortie  : ../data/processed/Loan_Default_Cameroun_Modele.csv (dataset pret pour la
          Semaine 2 : imputation, encodage, entrainement)

Objectif : retirer physiquement les colonnes sans aucune utilite (ni modele, ni
formulaire, ni description), listees dans
docs/Classification_Variables_Consolidee.txt (sections 1.B et 1.C) :
- fuite / post-decision banque : plafond_pret, approbation_anticipee, solvabilite,
  ecart_taux_interet, taux_interet, frais_initiaux_fcfa, amortissement_negatif,
  interet_seul, paiement_forfaitaire ;
- infrastructure de bureau de credit inexistante au Cameroun : type_credit_bureau,
  score_credit_bureau, type_credit_coemprunteur ;
- constante sans information : annee ;
- region_cameroun : decision revisee le 2026-07-31 (Aristide) - aucune valeur meme
  descriptive (couverture partielle 4/10 regions, gradient herite du remapping des
  regions US d'origine) -> supprimee completement, alors qu'elle etait jusque-la
  conservee en formulaire a titre descriptif uniquement.

In [19]:
import sys

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/ProjetScoringCredit" if IN_COLAB else ".."

INPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Enrichi.csv"
OUTPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Modele.csv"

df = pd.read_csv(INPUT_PATH)
print("Dimensions en entree :", df.shape)

Dimensions en entree : (148670, 32)


### 1. Colonnes de fuite / post-decision banque
Fixees ou connues par l'etablissement seulement apres (ou pendant) l'evaluation du
risque : jamais declarables par un nouveau demandeur au moment de la demande.

In [20]:
COLONNES_FUITE_POST_DECISION = [
    "plafond_pret",
    "approbation_anticipee",
    "solvabilite",
    "ecart_taux_interet",
    "taux_interet",
    "frais_initiaux_fcfa",
    "amortissement_negatif",
    "interet_seul",
    "paiement_forfaitaire",
]

### 2. Infrastructure de bureau de credit inexistante au Cameroun
Le public vise (secteur informel, primo-emprunteurs) n'a le plus souvent aucun
historique de bureau de credit individuel de type Experian/Equifax.

In [21]:
COLONNES_BUREAU_CREDIT = [
    "type_credit_bureau",
    "score_credit_bureau",
    "type_credit_coemprunteur",
]

### 3. Constante et geographie sans valeur
`annee` est constante (2019 sur toutes les lignes). `region_cameroun` etait
jusqu'ici conservee a titre descriptif ; decision revisee le 2026-07-31 : aucune
valeur meme descriptive, retiree entierement (dataset + formulaire).

In [22]:
COLONNES_SANS_VALEUR = [
    "annee",
    "region_cameroun",
]

COLONNES_A_RETIRER = COLONNES_FUITE_POST_DECISION + COLONNES_BUREAU_CREDIT + COLONNES_SANS_VALEUR

colonnes_absentes = [c for c in COLONNES_A_RETIRER if c not in df.columns]
assert not colonnes_absentes, f"Colonnes attendues mais deja absentes : {colonnes_absentes}"

df = df.drop(columns=COLONNES_A_RETIRER)
print(f"{len(COLONNES_A_RETIRER)} colonnes retirees : {COLONNES_A_RETIRER}")

14 colonnes retirees : ['plafond_pret', 'approbation_anticipee', 'solvabilite', 'ecart_taux_interet', 'taux_interet', 'frais_initiaux_fcfa', 'amortissement_negatif', 'interet_seul', 'paiement_forfaitaire', 'type_credit_bureau', 'score_credit_bureau', 'type_credit_coemprunteur', 'annee', 'region_cameroun']


### 4. Verification finale

In [23]:
print("Dimensions finales :", df.shape)
print("\nColonnes restantes :")
print(df.columns.tolist())

for col in COLONNES_A_RETIRER:
    assert col not in df.columns, f"{col} n'aurait pas du etre presente"

print("\nValeurs manquantes restantes (top 10) :")
print(df.isna().sum().sort_values(ascending=False).head(10))

Dimensions finales : (148670, 18)

Colonnes restantes :
['id_client', 'genre', 'type_pret', 'objet_pret', 'credit_ouvert', 'usage_professionnel', 'montant_pret_fcfa', 'duree_mois', 'revenu_mensuel_fcfa', 'tranche_age', 'mode_soumission', 'statut_remboursement', 'ratio_endettement', 'secteur_activite', 'niveau_education', 'membre_tontine', 'activite_saisonniere', 'utilisation_mobile_money']

Valeurs manquantes restantes (top 10) :
ratio_endettement      24121
revenu_mensuel_fcfa    10410
mode_soumission          200
tranche_age              200
objet_pret               134
duree_mois                41
genre                      0
id_client                  0
usage_professionnel        0
credit_ouvert              0
dtype: int64


### 5. Sauvegarde du dataset pret pour la Semaine 2

In [24]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset sauvegarde : {OUTPUT_PATH}")

Dataset sauvegarde : ../data/processed/Loan_Default_Cameroun_Modele.csv


---
## Etape 5 - Traitement des valeurs manquantes et aberrantes
*(source : `scripts/05_traitement_valeurs.py`)*

## Semaine 2 - Etape 1 : valeurs manquantes, valeurs aberrantes, recontextualisation
Entree  : ../data/processed/Loan_Default_Cameroun_Modele.csv (sortie du script 04)
Sortie  : ../data/processed/Loan_Default_Cameroun_Traite.csv

Taches (lundi 3 - mardi 4 aout 2026) :
1) Imputation des valeurs manquantes : mediane (numeriques), mode (categorielles)
2) Traitement des valeurs aberrantes par la methode IQR (capping/winsorisation)
3) Regeneration complete de duree_mois : le dataset source (pret immobilier
   americain) a 360 mois (30 ans) comme valeur dominante (81,8% des lignes), ce
   qui rend la methode IQR degeneree sur cette colonne (Q1 = Q3 = 360) et ne
   correspond a aucune realite de credit general/microfinance au Cameroun.
   Decision du 04/08/2026 (Aristide) : reouverture assumee du point tranche le
   29/07 (section 4.5 du plan, "duree_mois conservee telle quelle") - remplace
   par des durees tirees aleatoirement dans des tranches realistes, conditionnees
   sur objet_pret et montant_pret_fcfa (jamais sur statut_remboursement, pour ne
   pas fabriquer un proxy de la cible comme categorie_risque).
4) Suppression des lignes a revenu_mensuel_fcfa non plausible (< 15 000
   FCFA/mois, tres en-dessous du SMIG formel camerounais d'environ 41 875
   FCFA/mois) - decision du 04/08/2026, revisee le meme jour (suppression des
   lignes plutot que plancher/capping, pour ne pas conserver une valeur
   fabriquee dans le dataset d'entrainement).
5) Regeneration complete de tranche_age : le dataset source reflete une
   population de refinancement hypothecaire americain (0,9% de <25 ans, 18,8%
   de plus de 65 ans, defaut croissant avec l'age) plutot qu'une clientele
   active de credit general/microfinance camerounaise. Meme principe que
   duree_mois : tirage independant de statut_remboursement, decision du
   04/08/2026.
6) Relabellisation de type_pret (Type_1/2/3 -> Banque/Microfinance/
   Cooperative_epargne_credit) : simple renommage des categories (aucune valeur
   ne change de ligne), pour rattacher ce signal reel (22,8%/34,5%/25,1% de
   defaut) au contexte camerounais explicitement couvert par le perimetre du
   projet - banques traditionnelles ET institutions de microfinance (section 1
   du plan, tranche le 29/07). Decision du 04/08/2026.

In [25]:
import sys

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/ProjetScoringCredit" if IN_COLAB else ".."

INPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Modele.csv"
OUTPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Traite.csv"

RANDOM_SEED = 42

df = pd.read_csv(INPUT_PATH)
print("Dimensions en entree :", df.shape)
print("\nValeurs manquantes en entree :")
print(df.isna().sum()[df.isna().sum() > 0])

Dimensions en entree : (148670, 18)

Valeurs manquantes en entree :
objet_pret               134
duree_mois                41
revenu_mensuel_fcfa    10410
tranche_age              200
mode_soumission          200
ratio_endettement      24121
dtype: int64


### 1. Imputation des valeurs manquantes
Mode pour les variables categorielles, mediane pour les numeriques. `duree_mois`
est exclue de l'imputation par mediane : elle est integralement regeneree en
etape 3, ses valeurs manquantes sont donc traitees par la meme occasion.

In [26]:
COLONNES_CATEGORIELLES_A_IMPUTER = ["objet_pret", "tranche_age", "mode_soumission"]
COLONNES_NUMERIQUES_A_IMPUTER = ["revenu_mensuel_fcfa", "ratio_endettement"]

for col in COLONNES_CATEGORIELLES_A_IMPUTER:
    mode = df[col].mode(dropna=True)[0]
    n_manquantes = df[col].isna().sum()
    df[col] = df[col].fillna(mode)
    print(f"{col} : {n_manquantes} valeurs imputees par le mode ({mode!r})")

for col in COLONNES_NUMERIQUES_A_IMPUTER:
    mediane = df[col].median()
    n_manquantes = df[col].isna().sum()
    df[col] = df[col].fillna(mediane)
    print(f"{col} : {n_manquantes} valeurs imputees par la mediane ({mediane})")

objet_pret : 134 valeurs imputees par le mode ('Investissement_activite')


tranche_age : 200 valeurs imputees par le mode ('45-54')
mode_soumission : 200 valeurs imputees par le mode ('En_agence')
revenu_mensuel_fcfa : 10410 valeurs imputees par la mediane (115200.0)
ratio_endettement : 24121 valeurs imputees par la mediane (39.0)


### 2. Traitement des valeurs aberrantes (methode IQR)
Bornes `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` calculees sur les valeurs deja imputees.
Capping (winsorisation) plutot que suppression de lignes : dans un contexte de
scoring credit, les valeurs extremes de revenu/montant/endettement peuvent etre
un vrai signal de risque, pas seulement du bruit - les supprimer ferait perdre
des cas de defaut reels. `duree_mois` est exclue : elle est regeneree en etape 3.

In [27]:
COLONNES_IQR = ["montant_pret_fcfa", "revenu_mensuel_fcfa", "ratio_endettement"]

for col in COLONNES_IQR:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_aberrantes = ((df[col] < borne_basse) | (df[col] > borne_haute)).sum()
    df[col] = df[col].clip(lower=borne_basse, upper=borne_haute)
    print(
        f"{col} : bornes [{borne_basse:.0f}, {borne_haute:.0f}], "
        f"{n_aberrantes} valeurs cappees ({100 * n_aberrantes / len(df):.2f}%)"
    )

montant_pret_fcfa : bornes [-3270000, 15930000], 1895 valeurs cappees (1.27%)
revenu_mensuel_fcfa : bornes [-53400, 297000], 7823 valeurs cappees (5.26%)
ratio_endettement : bornes [16, 60], 5508 valeurs cappees (3.70%)


### 2 bis. Suppression des lignes a revenu non plausible (revenu_mensuel_fcfa)
La borne basse IQR est negative (aucun revenu n'est donc capte par la methode
statistique), mais un revenu declare sous 15 000 FCFA/mois n'est pas plausible
pour un demandeur de credit - a comparer au SMIG formel camerounais, d'environ
41 875 FCFA/mois depuis sa revalorisation de 2023 (36 270 FCFA avant). Seuil de
suppression fixe a 15 000 FCFA (nettement sous le SMIG, pas au niveau du SMIG) :
le public vise par le projet est en partie informel (agriculture, petit
commerce), ou des revenus reels sous le SMIG formel restent plausibles - relever
le seuil jusqu'au SMIG romprait avec ce principe (meme logique que l'exclusion
de score_credit_bureau, section 4 du plan). Decision du 04/08/2026 : suppression
des lignes plutot que capping/plancher, ces valeurs etant jugees trop peu
fiables pour etre corrigees et conservees dans le jeu d'entrainement.

In [28]:
SEUIL_REVENU_MIN_FCFA = 15_000
n_avant_suppression = len(df)
lignes_revenu_invalide = df["revenu_mensuel_fcfa"] < SEUIL_REVENU_MIN_FCFA
n_supprimees = lignes_revenu_invalide.sum()
df = df[~lignes_revenu_invalide].reset_index(drop=True)
print(
    f"revenu_mensuel_fcfa : {n_supprimees} lignes supprimees "
    f"(< {SEUIL_REVENU_MIN_FCFA} FCFA/mois) sur {n_avant_suppression} "
    f"({100 * n_supprimees / n_avant_suppression:.2f}%)"
)

revenu_mensuel_fcfa : 228 lignes supprimees (< 15000 FCFA/mois) sur 148670 (0.15%)


### 3. Regeneration de duree_mois avec des tranches realistes camerounaises
Tranches de montant (calibrees sur les quantiles reels de montant_pret_fcfa) et
durees en mois par objet_pret x tranche de montant, validees le 04/08/2026.
Tirage aleatoire (entier, loi uniforme) dans la fourchette correspondante, sans
jamais lire statut_remboursement : aucune fuite possible vers la cible.

In [29]:
def tranche_montant(montant):
    if montant < 2_000_000:
        return "petit"
    if montant < 8_000_000:
        return "moyen"
    if montant < 20_000_000:
        return "grand"
    return "tres_grand"


DUREE_BORNES_MOIS = {
    ("Autre", "petit"): (3, 12),
    ("Autre", "moyen"): (6, 18),
    ("Autre", "grand"): (12, 24),
    ("Autre", "tres_grand"): (18, 36),
    ("Achat", "petit"): (6, 12),
    ("Achat", "moyen"): (12, 24),
    ("Achat", "grand"): (24, 36),
    ("Achat", "tres_grand"): (24, 48),
    ("Investissement_activite", "petit"): (6, 18),
    ("Investissement_activite", "moyen"): (12, 36),
    ("Investissement_activite", "grand"): (24, 48),
    ("Investissement_activite", "tres_grand"): (36, 60),
    ("Refinancement", "petit"): (6, 12),
    ("Refinancement", "moyen"): (12, 24),
    ("Refinancement", "grand"): (18, 36),
    ("Refinancement", "tres_grand"): (24, 48),
}

rng = np.random.default_rng(RANDOM_SEED)
tranches = df["montant_pret_fcfa"].apply(tranche_montant)

bornes = pd.Series(
    list(zip(df["objet_pret"], tranches)), index=df.index
).map(DUREE_BORNES_MOIS)
assert bornes.isna().sum() == 0, "Combinaison objet_pret/tranche_montant non couverte"

bornes_basses = bornes.map(lambda b: b[0])
bornes_hautes = bornes.map(lambda b: b[1])
df["duree_mois"] = rng.integers(bornes_basses, bornes_hautes + 1)

print("Nouvelle distribution de duree_mois :")
print(df["duree_mois"].describe())
print("\nDuree mediane par objet_pret :")
print(df.groupby("objet_pret")["duree_mois"].median())

Nouvelle distribution de duree_mois :
count    148442.000000
mean         20.419255
std           8.962100
min           3.000000
25%          14.000000
50%          18.000000
75%          26.000000
max          48.000000
Name: duree_mois, dtype: float64

Duree mediane par objet_pret :
objet_pret
Achat                      21.0
Autre                      14.0
Investissement_activite    26.0
Refinancement              16.0
Name: duree_mois, dtype: float64


### 4. Regeneration de tranche_age avec une distribution active camerounaise
Distribution cible validee le 04/08/2026, recentree sur une population active
(25-44 ans = 57% au lieu de 35% dans le dataset source) plutot que sur un profil
de refinancement hypothecaire americain. Tirage aleatoire independant de toute
autre colonne (donc de statut_remboursement) : aucune correlation fabriquee avec
la cible.

Consequence assumee (verifiee le 04/08/2026) : le taux de defaut devient plat
sur toutes les tranches (24,0% a 25,1%, cf. verification finale ci-dessous),
puisque le tirage est independant de tout le reste. Decision d'Aristide : on
accepte cette perte de signal (deja artificiel, herite du profil americain) au
profit du realisme demographique. tranche_age passe donc de "feature ML"
(section 3.A de Classification_Variables_Consolidee.txt) a "descriptive
uniquement" (section 3.B), au meme titre que niveau_education - mise a jour du
document a faire en consequence.

In [30]:
DISTRIBUTION_AGE_CIBLE = {
    "<25": 0.08,
    "25-34": 0.30,
    "35-44": 0.27,
    "45-54": 0.18,
    "55-64": 0.11,
    "65-74": 0.04,
    ">74": 0.02,
}
assert abs(sum(DISTRIBUTION_AGE_CIBLE.values()) - 1.0) < 1e-9

tranches_age = list(DISTRIBUTION_AGE_CIBLE.keys())
probabilites_age = list(DISTRIBUTION_AGE_CIBLE.values())
df["tranche_age"] = rng.choice(tranches_age, size=len(df), p=probabilites_age)

print("Nouvelle distribution de tranche_age :")
print(df["tranche_age"].value_counts(normalize=True).reindex(tranches_age) * 100)

Nouvelle distribution de tranche_age :
tranche_age
<25       7.939801
25-34    30.146455
35-44    27.007855
45-54    17.867585
55-64    10.951079
65-74     4.051414
>74       2.035812
Name: proportion, dtype: float64


### 5. Relabellisation de type_pret (Banque / Microfinance / Cooperative)
Renommage pur des 3 categories existantes (Type_1/2/3), sans regeneration de
valeurs : aucune ligne ne change de categorie, donc aucun risque de fuite. Le
rattachement au type d'etablissement s'appuie sur l'ordre de risque deja observe
dans les donnees (Type_2 > Type_3 > Type_1 en taux de defaut), coherent avec des
profils reels camerounais (banque : dossiers plus formels/garantis, risque plus
faible ; microfinance : clientele plus informelle, risque plus eleve ;
cooperative d'epargne-credit : caution mutuelle entre membres, risque
intermediaire). Decision du 04/08/2026, perimetre banques + microfinance deja
valide le 29/07 (section 1 du plan).

In [31]:
LIBELLES_TYPE_PRET = {
    "Type_1": "Banque",
    "Type_2": "Microfinance",
    "Type_3": "Cooperative_epargne_credit",
}
df["type_pret"] = df["type_pret"].map(LIBELLES_TYPE_PRET)
assert df["type_pret"].isna().sum() == 0, "Valeur de type_pret non couverte par le mapping"

print("Nouvelles categories de type_pret :")
print(df["type_pret"].value_counts())

Nouvelles categories de type_pret :
type_pret
Banque                        113029
Microfinance                   20692
Cooperative_epargne_credit     14721
Name: count, dtype: int64


### 6. Verification finale

In [32]:
print("Dimensions finales :", df.shape)
assert df.isna().sum().sum() == 0, "Des valeurs manquantes subsistent"
print("Aucune valeur manquante restante : OK")

print("\nTaux de defaut par tranche_age (verification de la perte de signal assumee) :")
print(
    (df.groupby("tranche_age")["statut_remboursement"].apply(lambda s: (s == "Defaut").mean() * 100))
    .round(2)
)

Dimensions finales : (148442, 18)
Aucune valeur manquante restante : OK

Taux de defaut par tranche_age (verification de la perte de signal assumee) :
tranche_age
25-34    24.52
35-44    24.65
45-54    24.17
55-64    24.56
65-74    24.79
<25      24.76
>74      25.71
Name: statut_remboursement, dtype: float64


### 7. Sauvegarde du dataset traite

In [33]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset sauvegarde : {OUTPUT_PATH}")

Dataset sauvegarde : ../data/processed/Loan_Default_Cameroun_Traite.csv


---
## Etape 6 - Encodage des variables categorielles et normalisation
*(source : `scripts/06_encodage_normalisation.py`)*

## Semaine 2 - Etape 2 : encodage des variables categorielles et normalisation
Entree  : ../data/processed/Loan_Default_Cameroun_Traite.csv (sortie du script 05)
Sortie  : ../data/processed/Loan_Default_Cameroun_Encode.csv

Taches (mercredi 5 - jeudi 6 aout 2026, mise a jour dimanche 9 aout 2026) :
1) Suppression de mode_soumission (decision du 06/08/2026, voir note ci-dessous)
   et de type_pret (decision du 09/08/2026, voir note ci-dessous)
2) Encodage de la cible statut_remboursement (binaire, 0/1)
3) Encodage binaire (Oui/Non -> 1/0) de usage_professionnel et credit_ouvert
4) One-Hot Encoding de objet_pret, secteur_activite
5) Normalisation (StandardScaler) des variables numeriques du modele

Note du 06/08/2026 (Aristide) - suppression de mode_soumission :
la liste finale d'Andy (Liste_finale.docx, categorie A section 4) et sa
documentation Power BI recommandent mode_soumission comme variable prioritaire
(28,4% de defaut en agence contre 17,5% en ligne). Decision revisee malgre ce
signal reel confirme le 30/07 : la configuration retenue pour l'application ne
traite que des demandes de credit physique en agence (perimetre produit, pas un
constat statistique) - la colonne serait donc constante ("En_agence") pour
toute nouvelle demande en production, sans aucune valeur predictive ni meme
descriptive a ce stade. Meme principe que la suppression de region_cameroun le
31/07 (section 1.C de Classification_Variables_Consolidee.txt) : une colonne
retiree du perimetre applicatif est supprimee du dataset dans son ensemble,
pas seulement exclue des features du modele. A signaler a Andy avant le point
d'equipe du dimanche 9 aout (sa liste finale devra etre mise a jour en
consequence).

Note du 06/08/2026 (Aristide) - divergences avec la liste finale d'Andy :
la liste d'Andy reintroduit genre et tranche_age comme features ML, et exclut
credit_ouvert. Les deux premiers points contredisent des decisions deja
tranchees et documentees (genre : motif ethique, 29/07 ; tranche_age :
regeneree independamment de la cible pour eviter la fuite, reclassee
descriptive le 04/08). Le troisieme (credit_ouvert) contredit le statut de
feature ML retenu le 30/07 (signal reel mais rare, a surveiller). Ce script
suit la classification consolidee (Classification_Variables_Consolidee.txt),
pas la liste d'Andy, sur ces 3 points precis - egalement a signaler avant le
9 aout.

Note du 09/08/2026 (decision d'equipe, point du 9 aout) - suppression de
type_pret : signal statistique reel confirme le 30/07 (22,8%/34,5%/25,1% de
defaut selon Banque/Microfinance/Cooperative_epargne_credit), conserve comme
feature ML (One-Hot) jusqu'a cette date. Decision d'equipe : le projet se
positionne deja comme une solution de microfinance de base (recadrage de
perimetre produit, pas un constat statistique) - la colonne serait donc
constante ("Microfinance") pour toute nouvelle demande en production, sans
aucune valeur meme descriptive. Meme principe que mode_soumission ci-dessus et
region_cameroun (31/07) : supprimee du dataset dans son ensemble, pas
seulement exclue des features du modele. Voir Classification_Variables_
Consolidee.txt section 1.E.

In [34]:
import sys

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/ProjetScoringCredit" if IN_COLAB else ".."

INPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Traite.csv"
OUTPUT_PATH = f"{BASE_DIR}/data/processed/Loan_Default_Cameroun_Encode.csv"

df = pd.read_csv(INPUT_PATH)
print("Dimensions en entree :", df.shape)

Dimensions en entree : (148442, 18)


### 1. Suppression de mode_soumission et type_pret
Colonnes retirees du perimetre applicatif (voir notes ci-dessus) : supprimees
du dataset dans son ensemble, pas seulement des features du modele.

In [35]:
df = df.drop(columns=["mode_soumission", "type_pret"])
print("mode_soumission et type_pret supprimees. Dimensions :", df.shape)

mode_soumission et type_pret supprimees. Dimensions : (148442, 16)


### 2. Encodage de la cible (statut_remboursement)
Encodage binaire simple : 1 = Defaut (classe positive, celle que le modele doit
detecter), 0 = Rembourse. La colonne est ecrasee en place (meme nom), il n'y a
pas de doublon a gerer (statut_remboursement_label a deja ete supprime au
script 03).

In [36]:
MAPPING_CIBLE = {"Rembourse": 0, "Defaut": 1}
valeurs_avant = set(df["statut_remboursement"].unique())
assert valeurs_avant == set(MAPPING_CIBLE), f"Valeurs inattendues : {valeurs_avant}"
df["statut_remboursement"] = df["statut_remboursement"].map(MAPPING_CIBLE)
print("Cible encodee :", MAPPING_CIBLE)
print(df["statut_remboursement"].value_counts(normalize=True).round(4) * 100)

Cible encodee : {'Rembourse': 0, 'Defaut': 1}
statut_remboursement
0    75.45
1    24.55
Name: proportion, dtype: float64


### 3. Colonnes exclues des features du modele (conservees telles quelles)
Ces colonnes restent dans le fichier de sortie pour le profil client / le
rapport, mais ne sont ni encodees ni normalisees : elles ne sont pas destinees
a alimenter l'entrainement (voir Classification_Variables_Consolidee.txt,
section 3.B). id_client suit la meme logique (identifiant technique, jamais
une feature).

In [37]:
COLONNES_NON_FEATURES = [
    "id_client",
    "genre",
    "tranche_age",
    "niveau_education",
    "membre_tontine",
    "activite_saisonniere",
    "utilisation_mobile_money",
]
print("Colonnes conservees sans transformation :", COLONNES_NON_FEATURES)

Colonnes conservees sans transformation : ['id_client', 'genre', 'tranche_age', 'niveau_education', 'membre_tontine', 'activite_saisonniere', 'utilisation_mobile_money']


### 4. Encodage binaire (Oui/Non -> 1/0)
usage_professionnel et credit_ouvert n'ont que 2 categories sans ordre naturel
a preserver : un encodage binaire direct suffit, pas besoin de One-Hot (qui
ne ferait que dupliquer l'information sur 2 colonnes).

In [38]:
COLONNES_BINAIRES = ["usage_professionnel", "credit_ouvert"]
MAPPING_BINAIRE = {"Non": 0, "Oui": 1}

for col in COLONNES_BINAIRES:
    valeurs = set(df[col].unique())
    assert valeurs == set(MAPPING_BINAIRE), f"{col} : valeurs inattendues {valeurs}"
    df[col] = df[col].map(MAPPING_BINAIRE)
    print(f"{col} encodee (Non=0, Oui=1)")

usage_professionnel encodee (Non=0, Oui=1)


credit_ouvert encodee (Non=0, Oui=1)


### 5. One-Hot Encoding des variables categorielles nominales
objet_pret et secteur_activite n'ont pas d'ordre naturel entre leurs
categories : le One-Hot Encoding evite d'imposer une hierarchie artificielle
qu'un Label Encoding introduirait.

In [39]:
COLONNES_ONEHOT = ["objet_pret", "secteur_activite"]

for col in COLONNES_ONEHOT:
    print(f"{col} : {df[col].nunique()} categories -> {sorted(df[col].unique())}")

df = pd.get_dummies(df, columns=COLONNES_ONEHOT, prefix=COLONNES_ONEHOT, dtype=int)
print("\nDimensions apres One-Hot Encoding :", df.shape)

objet_pret : 4 categories -> ['Achat', 'Autre', 'Investissement_activite', 'Refinancement']
secteur_activite : 6 categories -> ['Agriculture', 'Artisanat', 'Commerce/NÃ©goce', 'Petit commerce', 'Profession libÃ©rale', 'SalariÃ© formel']



Dimensions apres One-Hot Encoding : (148442, 24)


### 6. Normalisation des variables numeriques (StandardScaler)
Centrage-reduction (moyenne 0, ecart-type 1) : necessaire pour les modeles
sensibles a l'echelle des variables (regression logistique) et sans effet
negatif pour les modeles bases sur des arbres (Random Forest, XGBoost) prevus
en semaine 3 - un seul dataset encode sert donc aux 3 modeles.

In [40]:
COLONNES_NUMERIQUES = [
    "montant_pret_fcfa",
    "duree_mois",
    "revenu_mensuel_fcfa",
    "ratio_endettement",
]

scaler = StandardScaler()
df[COLONNES_NUMERIQUES] = scaler.fit_transform(df[COLONNES_NUMERIQUES])
print("Variables normalisees :", COLONNES_NUMERIQUES)
print(df[COLONNES_NUMERIQUES].describe().round(3))

Variables normalisees : ['montant_pret_fcfa', 'duree_mois', 'revenu_mensuel_fcfa', 'ratio_endettement']
       montant_pret_fcfa  duree_mois  revenu_mensuel_fcfa  ratio_endettement
count         148442.000  148442.000           148442.000         148442.000
mean               0.000       0.000                0.000              0.000
std                1.000       1.000                1.000              1.000
min               -1.844      -1.944               -1.653             -2.372
25%               -0.778      -0.716               -0.735             -0.563
50%               -0.186      -0.270               -0.216              0.095
75%                0.643       0.623                0.512              0.643
max                2.775       3.077                2.407              2.452


### 7. Verification finale

In [41]:
print("Dimensions finales :", df.shape)
assert df.isna().sum().sum() == 0, "Des valeurs manquantes subsistent"
print("Aucune valeur manquante restante : OK")

colonnes_features = [
    c
    for c in df.columns
    if c not in COLONNES_NON_FEATURES and c != "statut_remboursement"
]
print(f"\n{len(colonnes_features)} colonnes de features pretes pour la modelisation :")
print(colonnes_features)

Dimensions finales : (148442, 24)
Aucune valeur manquante restante : OK

16 colonnes de features pretes pour la modelisation :
['credit_ouvert', 'usage_professionnel', 'montant_pret_fcfa', 'duree_mois', 'revenu_mensuel_fcfa', 'ratio_endettement', 'objet_pret_Achat', 'objet_pret_Autre', 'objet_pret_Investissement_activite', 'objet_pret_Refinancement', 'secteur_activite_Agriculture', 'secteur_activite_Artisanat', 'secteur_activite_Commerce/NÃ©goce', 'secteur_activite_Petit commerce', 'secteur_activite_Profession libÃ©rale', 'secteur_activite_SalariÃ© formel']


### 8. Sauvegarde du dataset encode

In [42]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset sauvegarde : {OUTPUT_PATH}")

Dataset sauvegarde : ../data/processed/Loan_Default_Cameroun_Encode.csv
